## Import libs

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.python.data import Dataset
from tensorflow.keras.optimizers import Adam
import seaborn as sns


from falsb4mpa.modeling.zhang.learning.multi_adv import train_loop as zhang_train
from falsb4mpa.dataset.load_data import load_data
from falsb4mpa.evaluation.evaluation import compute_predictive_metrics, compute_fair_metrics, compute_adv_metrics, compute_tradeoff, fair_evaluation, compute_intersectional_fair_metrics
from falsb4mpa.modeling.zhang.models.multi_adv import ZhangMultAdv

## Preliminaries

In [2]:
batch_size = 64
epochs = 100
learning_rate = 0.001

In [3]:
# cv_seeds = [13]
cv_seeds = [13, 29, 42, 55, 73]

## Load data

In [4]:
data_name = 'german-mpa-bin-wout-agg'

In [5]:
x, y, a1, a2 = load_data(data_name)
raw_data = (x, y, a1, a2)

In [6]:
xdim = x.shape[1]
ydim = y.shape[1]
a1dim = a1.shape[1]
a2dim = a2.shape[1]
zdim = 8
print(xdim, ydim, a1dim, a2dim, zdim)

28 1 1 1 8


## Result file

In [7]:
header = [
    "model_name", "cv_seed", 
    "clas_acc", "f1-micro", "f1-macro",
    "a1_dp", "a1_deqodds", "a1_deqopp", 
    "a1_TN_g0", "a1_FP_g0", "a1_FN_g0", "a1_TP_g0", "a1_TN_g1", "a1_FP_g1", "a1_FN_g1", "a1_TP_g1", 
    "a2_dp", "a2_deqodds", "a2_deqopp", 
    "a2_TN_g0", "a2_FP_g0", "a2_FN_g0", "a2_TP_g0", "a2_TN_g1", "a2_FP_g1", "a2_FN_g1", "a2_TP_g1",
    "wc_spd", "wc_aod", "wc_eod",
    "last_cosine_similarity"
]

results = []

## Testing

### DemPar

In [8]:
fairdef = "DemPar"

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test = train_test_split(
        x, y, a1, a2, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a1_train, a2_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a1_test, a2_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    opt = Adam(learning_rate=learning_rate)

    model = ZhangMultAdv(xdim=xdim, ydim=ydim, a1dim=a1dim, a2dim=a2dim, batch_size=batch_size, fairdef=fairdef)
    
    ret, dULa1, dULa2, cos_sim = zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A1, A2, Y_hat, A1_hat, A2_hat = fair_evaluation(model, test_data)
    
    clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix = compute_predictive_metrics(Y, Y_hat)
    
    adv1_acc = compute_adv_metrics(A1, A1_hat)
    adv2_acc = compute_adv_metrics(A2, A2_hat)
    
    a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1 = compute_fair_metrics(Y, A1, Y_hat, a1dim)
    a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1 = compute_fair_metrics(Y, A2, Y_hat, a2dim)

    wc_spd, wc_aod, wc_eod = compute_intersectional_fair_metrics(Y, A1, A2, Y_hat, a1dim, a2dim)


    # fair_metrics = (dp, deqodds, deqopp)
    # tradeoff = []
    # for fair_metric in fair_metrics:
    #     tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    # result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    result = ['MultAdvBin4DP', cv_seed]
    result += [clas_acc, clas_f1_micro, clas_f1_macro]
    result += [a1_dp, a1_deqodds, a1_deqopp] + a1_metrics_g0 + a1_metrics_g1 
    result += [a2_dp, a2_deqodds, a2_deqopp] + a2_metrics_g0 + a2_metrics_g1
    result += [wc_spd, wc_aod, wc_eod]
    result += [cos_sim]


    results.append(result)

    del(opt, x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test, train_data, test_data, model, ret)
    del(Y, A1, A2, Y_hat, A1_hat, A2_hat)
    del(clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix, adv1_acc, adv2_acc)
    del(a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1, a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1)
    del(wc_spd, wc_aod, wc_eod)
    del(cos_sim)

2026-05-14 17:13:44.208720: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 1 | Clf loss/acc 0.59/0.67 | Adv1 loss/acc 1.04/0.30 | Adv2 loss/acc 0.55/0.58 | Cos Sim -0.24


2026-05-14 17:13:44.531731: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 2 | Clf loss/acc 0.58/0.67 | Adv1 loss/acc 1.03/0.30 | Adv2 loss/acc 0.55/0.58 | Cos Sim -0.25
> Epoch: 3 | Clf loss/acc 0.58/0.67 | Adv1 loss/acc 1.02/0.30 | Adv2 loss/acc 0.55/0.58 | Cos Sim -0.25


2026-05-14 17:13:45.228843: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


...

> Epoch: 64 | Clf loss/acc 0.46/0.74 | Adv1 loss/acc 0.91/0.33 | Adv2 loss/acc 0.57/0.57 | Cos Sim -0.33
> Epoch: 65 | Clf loss/acc 0.46/0.74 | Adv1 loss/acc 0.91/0.33 | Adv2 loss/acc 0.57/0.57 | Cos Sim -0.33
> Epoch: 66 | Clf loss/acc 0.46/0.75 | Adv1 loss/acc 0.91/0.33 | Adv2 loss/acc 0.57/0.56 | Cos Sim -0.33
> Epoch: 67 | Clf loss/acc 0.46/0.75 | Adv1 loss/acc 0.91/0.33 | Adv2 loss/acc 0.57/0.57 | Cos Sim -0.33
> Epoch: 68 | Clf loss/acc 0.46/0.75 | Adv1 loss/acc 0.91/0.33 | Adv2 loss/acc 0.57/0.57 | Cos Sim -0.33
> Epoch: 69 | Clf loss/acc 0.46/0.75 | Adv1 loss/acc 0.91/0.33 | Adv2 loss/acc 0.57/0.57 | Cos Sim -0.33
> Epoch: 70 | Clf loss/acc 0.46/0.75 | Adv1 loss/acc 0.91/0.33 | Adv2 loss/acc 0.57/0.57 | Cos Sim -0.33
> Epoch: 71 | Clf loss/acc 0.46/0.75 | Adv1 loss/acc 0.91/0.33 | Adv2 loss/acc 0.57/0.57 | Cos Sim -0.33
> Epoch: 72 | Clf loss/acc 0.46/0.75 | Adv1 loss/acc 0.90/0.33 | Adv2 loss/acc 0.57/0.57 | Cos Sim -0.33
> Epoch: 73 | Clf loss/acc 0.46/0.76 | Adv1 loss/acc 0.

2026-05-14 17:14:25.235798: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 27 | Clf loss/acc 0.58/0.72 | Adv1 loss/acc 0.99/0.31 | Adv2 loss/acc 0.70/0.58 | Cos Sim -0.09
> Epoch: 28 | Clf loss/acc 0.57/0.73 | Adv1 loss/acc 0.99/0.31 | Adv2 loss/acc 0.70/0.58 | Cos Sim -0.09
> Epoch: 29 | Clf loss/acc 0.57/0.73 | Adv1 loss/acc 0.99/0.31 | Adv2 loss/acc 0.70/0.58 | Cos Sim -0.09
> Epoch: 30 | Clf loss/acc 0.57/0.73 | Adv1 loss/acc 0.99/0.31 | Adv2 loss/acc 0.70/0.58 | Cos Sim -0.09
> Epoch: 31 | Clf loss/acc 0.57/0.73 | Adv1 loss/acc 0.99/0.31 | Adv2 loss/acc 0.70/0.58 | Cos Sim -0.10
> Epoch: 32 | Clf loss/acc 0.57/0.73 | Adv1 loss/acc 0.98/0.31 | Adv2 loss/acc 0.70/0.58 | Cos Sim -0.10
> Epoch: 33 | Clf loss/acc 0.57/0.73 | Adv1 loss/acc 0.98/0.31 | Adv2 loss/acc 0.70/0.58 | Cos Sim -0.10
> Epoch: 34 | Clf loss/acc 0.57/0.73 | Adv1 loss/acc 0.98/0.31 | Adv2 loss/acc 0.70/0.58 | Cos Sim -0.10
> Epoch: 35 | Clf loss/acc 0.56/0.74 | Adv1 loss/acc 0.98/0.31 | Adv2 loss/acc 0.70/0.58 | Cos Sim -0.10
> Epoch: 36 | Clf loss/acc 0.56/0.74 | Adv1 loss/acc 0.

2026-05-14 17:15:05.701537: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 54 | Clf loss/acc 0.53/0.74 | Adv1 loss/acc 0.88/0.31 | Adv2 loss/acc 0.75/0.59 | Cos Sim 0.05
> Epoch: 55 | Clf loss/acc 0.53/0.74 | Adv1 loss/acc 0.88/0.31 | Adv2 loss/acc 0.75/0.59 | Cos Sim 0.05
> Epoch: 56 | Clf loss/acc 0.53/0.74 | Adv1 loss/acc 0.88/0.31 | Adv2 loss/acc 0.75/0.58 | Cos Sim 0.05
> Epoch: 57 | Clf loss/acc 0.53/0.74 | Adv1 loss/acc 0.88/0.31 | Adv2 loss/acc 0.75/0.58 | Cos Sim 0.06
> Epoch: 58 | Clf loss/acc 0.53/0.75 | Adv1 loss/acc 0.87/0.31 | Adv2 loss/acc 0.75/0.58 | Cos Sim 0.06
> Epoch: 59 | Clf loss/acc 0.53/0.75 | Adv1 loss/acc 0.87/0.31 | Adv2 loss/acc 0.75/0.58 | Cos Sim 0.06
> Epoch: 60 | Clf loss/acc 0.53/0.75 | Adv1 loss/acc 0.87/0.31 | Adv2 loss/acc 0.75/0.59 | Cos Sim 0.06
> Epoch: 61 | Clf loss/acc 0.53/0.75 | Adv1 loss/acc 0.87/0.32 | Adv2 loss/acc 0.75/0.59 | Cos Sim 0.06
> Epoch: 62 | Clf loss/acc 0.53/0.75 | Adv1 loss/acc 0.87/0.32 | Adv2 loss/acc 0.75/0.59 | Cos Sim 0.06
> Epoch: 63 | Clf loss/acc 0.53/0.75 | Adv1 loss/acc 0.87/0.32 |

### EqOdds

In [9]:
fairdef = "EqOdds"

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test = train_test_split(
        x, y, a1, a2, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a1_train, a2_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a1_test, a2_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    opt = Adam(learning_rate=learning_rate)

    model = ZhangMultAdv(xdim=xdim, ydim=ydim, a1dim=a1dim, a2dim=a2dim, batch_size=batch_size, fairdef=fairdef)
    
    ret, dULa1, dULa2, cos_sim = zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A1, A2, Y_hat, A1_hat, A2_hat = fair_evaluation(model, test_data)
    
    clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix = compute_predictive_metrics(Y, Y_hat)
    
    adv1_acc = compute_adv_metrics(A1, A1_hat)
    adv2_acc = compute_adv_metrics(A2, A2_hat)
    
    a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1 = compute_fair_metrics(Y, A1, Y_hat, a1dim)
    a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1 = compute_fair_metrics(Y, A2, Y_hat, a2dim)

    wc_spd, wc_aod, wc_eod = compute_intersectional_fair_metrics(Y, A1, A2, Y_hat, a1dim, a2dim)


    # fair_metrics = (dp, deqodds, deqopp)
    # tradeoff = []
    # for fair_metric in fair_metrics:
    #     tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    # result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    result = ['MultAdvBin4EqOdds', cv_seed]
    result += [clas_acc, clas_f1_micro, clas_f1_macro]
    result += [a1_dp, a1_deqodds, a1_deqopp] + a1_metrics_g0 + a1_metrics_g1 
    result += [a2_dp, a2_deqodds, a2_deqopp] + a2_metrics_g0 + a2_metrics_g1
    result += [wc_spd, wc_aod, wc_eod]
    result += [cos_sim]


    results.append(result)

    del(opt, x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test, train_data, test_data, model, ret)
    del(Y, A1, A2, Y_hat, A1_hat, A2_hat)
    del(clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix, adv1_acc, adv2_acc)
    del(a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1, a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1)
    del(wc_spd, wc_aod, wc_eod)
    del(cos_sim)

> Epoch: 1 | Clf loss/acc 0.59/0.67 | Adv1 loss/acc 1.03/0.30 | Adv2 loss/acc 0.55/0.58 | Cos Sim -0.24
> Epoch: 2 | Clf loss/acc 0.58/0.67 | Adv1 loss/acc 1.02/0.30 | Adv2 loss/acc 0.55/0.58 | Cos Sim -0.25
> Epoch: 3 | Clf loss/acc 0.58/0.67 | Adv1 loss/acc 1.01/0.30 | Adv2 loss/acc 0.55/0.58 | Cos Sim -0.26
> Epoch: 4 | Clf loss/acc 0.57/0.67 | Adv1 loss/acc 1.01/0.30 | Adv2 loss/acc 0.55/0.58 | Cos Sim -0.26
> Epoch: 5 | Clf loss/acc 0.56/0.67 | Adv1 loss/acc 1.00/0.30 | Adv2 loss/acc 0.55/0.58 | Cos Sim -0.27
> Epoch: 6 | Clf loss/acc 0.56/0.67 | Adv1 loss/acc 0.99/0.30 | Adv2 loss/acc 0.55/0.58 | Cos Sim -0.27


2026-05-14 17:16:26.360631: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 7 | Clf loss/acc 0.55/0.67 | Adv1 loss/acc 0.99/0.30 | Adv2 loss/acc 0.55/0.58 | Cos Sim -0.28
> Epoch: 8 | Clf loss/acc 0.55/0.67 | Adv1 loss/acc 0.98/0.30 | Adv2 loss/acc 0.55/0.58 | Cos Sim -0.28
> Epoch: 9 | Clf loss/acc 0.54/0.67 | Adv1 loss/acc 0.98/0.30 | Adv2 loss/acc 0.55/0.58 | Cos Sim -0.29
> Epoch: 10 | Clf loss/acc 0.54/0.67 | Adv1 loss/acc 0.97/0.30 | Adv2 loss/acc 0.55/0.58 | Cos Sim -0.29
> Epoch: 11 | Clf loss/acc 0.53/0.68 | Adv1 loss/acc 0.96/0.30 | Adv2 loss/acc 0.55/0.58 | Cos Sim -0.29
> Epoch: 12 | Clf loss/acc 0.53/0.68 | Adv1 loss/acc 0.96/0.30 | Adv2 loss/acc 0.55/0.58 | Cos Sim -0.30
> Epoch: 13 | Clf loss/acc 0.53/0.68 | Adv1 loss/acc 0.95/0.30 | Adv2 loss/acc 0.55/0.58 | Cos Sim -0.30
> Epoch: 14 | Clf loss/acc 0.52/0.69 | Adv1 loss/acc 0.95/0.30 | Adv2 loss/acc 0.55/0.58 | Cos Sim -0.31
> Epoch: 15 | Clf loss/acc 0.52/0.69 | Adv1 loss/acc 0.94/0.30 | Adv2 loss/acc 0.56/0.58 | Cos Sim -0.31
> Epoch: 16 | Clf loss/acc 0.52/0.70 | Adv1 loss/acc 0.93/

### EqOpp

In [10]:
fairdef = "EqOpp"

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test = train_test_split(
        x, y, a1, a2, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a1_train, a2_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a1_test, a2_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    opt = Adam(learning_rate=learning_rate)

    model = ZhangMultAdv(xdim=xdim, ydim=ydim, a1dim=a1dim, a2dim=a2dim, batch_size=batch_size, fairdef=fairdef)
    
    ret, dULa1, dULa2, cos_sim = zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A1, A2, Y_hat, A1_hat, A2_hat = fair_evaluation(model, test_data)
    
    clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix = compute_predictive_metrics(Y, Y_hat)
    
    adv1_acc = compute_adv_metrics(A1, A1_hat)
    adv2_acc = compute_adv_metrics(A2, A2_hat)
    
    a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1 = compute_fair_metrics(Y, A1, Y_hat, a1dim)
    a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1 = compute_fair_metrics(Y, A2, Y_hat, a2dim)

    wc_spd, wc_aod, wc_eod = compute_intersectional_fair_metrics(Y, A1, A2, Y_hat, a1dim, a2dim)


    # fair_metrics = (dp, deqodds, deqopp)
    # tradeoff = []
    # for fair_metric in fair_metrics:
    #     tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    # result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    result = ['MultAdvBin4EqOpp', cv_seed]
    result += [clas_acc, clas_f1_micro, clas_f1_macro]
    result += [a1_dp, a1_deqodds, a1_deqopp] + a1_metrics_g0 + a1_metrics_g1 
    result += [a2_dp, a2_deqodds, a2_deqopp] + a2_metrics_g0 + a2_metrics_g1
    result += [wc_spd, wc_aod, wc_eod]
    result += [cos_sim]


    results.append(result)

    del(opt, x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test, train_data, test_data, model, ret)
    del(Y, A1, A2, Y_hat, A1_hat, A2_hat)
    del(clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix, adv1_acc, adv2_acc)
    del(a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1, a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1)
    del(wc_spd, wc_aod, wc_eod)
    del(cos_sim)

> Epoch: 1 | Clf loss/acc 0.59/0.67 | Adv1 loss/acc 0.76/0.30 | Adv2 loss/acc 0.38/0.58 | Cos Sim -0.23
> Epoch: 2 | Clf loss/acc 0.58/0.67 | Adv1 loss/acc 0.75/0.30 | Adv2 loss/acc 0.38/0.58 | Cos Sim -0.24
> Epoch: 3 | Clf loss/acc 0.58/0.67 | Adv1 loss/acc 0.75/0.30 | Adv2 loss/acc 0.38/0.58 | Cos Sim -0.25
> Epoch: 4 | Clf loss/acc 0.57/0.67 | Adv1 loss/acc 0.74/0.30 | Adv2 loss/acc 0.38/0.58 | Cos Sim -0.25
> Epoch: 5 | Clf loss/acc 0.56/0.67 | Adv1 loss/acc 0.74/0.30 | Adv2 loss/acc 0.38/0.58 | Cos Sim -0.26
> Epoch: 6 | Clf loss/acc 0.56/0.67 | Adv1 loss/acc 0.73/0.30 | Adv2 loss/acc 0.38/0.58 | Cos Sim -0.26
> Epoch: 7 | Clf loss/acc 0.55/0.67 | Adv1 loss/acc 0.73/0.30 | Adv2 loss/acc 0.38/0.58 | Cos Sim -0.27
> Epoch: 8 | Clf loss/acc 0.55/0.67 | Adv1 loss/acc 0.73/0.30 | Adv2 loss/acc 0.38/0.58 | Cos Sim -0.27
> Epoch: 9 | Clf loss/acc 0.54/0.67 | Adv1 loss/acc 0.72/0.30 | Adv2 loss/acc 0.38/0.58 | Cos Sim -0.28
> Epoch: 10 | Clf loss/acc 0.54/0.67 | Adv1 loss/acc 0.72/0.30 |

2026-05-14 17:19:10.771378: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 14 | Clf loss/acc 0.52/0.68 | Adv1 loss/acc 0.70/0.30 | Adv2 loss/acc 0.39/0.58 | Cos Sim -0.30
> Epoch: 15 | Clf loss/acc 0.52/0.69 | Adv1 loss/acc 0.70/0.30 | Adv2 loss/acc 0.39/0.58 | Cos Sim -0.31
> Epoch: 16 | Clf loss/acc 0.52/0.69 | Adv1 loss/acc 0.69/0.30 | Adv2 loss/acc 0.39/0.58 | Cos Sim -0.31
> Epoch: 17 | Clf loss/acc 0.51/0.70 | Adv1 loss/acc 0.69/0.30 | Adv2 loss/acc 0.39/0.58 | Cos Sim -0.32
> Epoch: 18 | Clf loss/acc 0.51/0.70 | Adv1 loss/acc 0.68/0.30 | Adv2 loss/acc 0.39/0.58 | Cos Sim -0.32
> Epoch: 19 | Clf loss/acc 0.51/0.70 | Adv1 loss/acc 0.68/0.30 | Adv2 loss/acc 0.39/0.58 | Cos Sim -0.33
> Epoch: 20 | Clf loss/acc 0.51/0.71 | Adv1 loss/acc 0.68/0.30 | Adv2 loss/acc 0.39/0.58 | Cos Sim -0.34
> Epoch: 21 | Clf loss/acc 0.50/0.71 | Adv1 loss/acc 0.67/0.30 | Adv2 loss/acc 0.39/0.58 | Cos Sim -0.34
> Epoch: 22 | Clf loss/acc 0.50/0.71 | Adv1 loss/acc 0.67/0.30 | Adv2 loss/acc 0.39/0.58 | Cos Sim -0.34
> Epoch: 23 | Clf loss/acc 0.50/0.72 | Adv1 loss/acc 0.

## Saving into DF then CSV

In [11]:
result_df = pd.DataFrame(results, columns=header)
result_df

,model_name,cv_seed,clas_acc,f1-micro,f1-macro,a1_dp,a1_deqodds,a1_deqopp,a1_TN_g0,a1_FP_g0,...,a2_FN_g0,a2_TP_g0,a2_TN_g1,a2_FP_g1,a2_FN_g1,a2_TP_g1,wc_spd,wc_aod,wc_eod,last_cosine_similarity
0,MultAdvBin4DP,13,0.699219,0.699219,0.592517,0.799834,0.758602,0.879841,15.0,11.0,...,17.0,98.0,11.0,19.0,16.0,57.0,0.759553,0.714494,0.843129,-0.323780
1,MultAdvBin4DP,29,0.707031,0.707031,0.626277,0.952943,0.922923,0.932447,11.0,25.0,...,8.0,91.0,19.0,24.0,11.0,59.0,0.837366,0.809402,0.720000,-0.157530
2,MultAdvBin4DP,42,0.703125,0.703125,0.576454,0.850609,0.837762,0.873077,9.0,13.0,...,8.0,101.0,11.0,19.0,14.0,59.0,0.802769,0.689437,0.798754,0.077006
3,MultAdvBin4DP,55,0.664062,0.664062,0.542401,0.766697,0.757584,0.784399,11.0,15.0,...,13.0,100.0,11.0,19.0,14.0,51.0,0.672304,0.645276,0.714729,-0.080856
4,MultAdvBin4DP,73,0.738281,0.738281,0.628865,0.986627,0.966034,0.995704,9.0,21.0,...,8.0,109.0,14.0,25.0,10.0,55.0,0.843305,0.826062,0.885457,0.159518
5,MultAdvBin4EqOdds,13,0.703125,0.703125,0.599967,0.805648,0.770507,0.879841,15.0,11.0,...,18.0,97.0,12.0,18.0,15.0,58.0,0.768173,0.720112,0.854365,-0.328490
6,MultAdvBin4EqOdds,29,0.710938,0.710938,0.625997,0.917686,0.948483,0.916574,12.0,24.0,...,6.0,93.0,19.0,24.0,11.0,59.0,0.803571,0.830496,0.720000,-0.230353
7,MultAdvBin4EqOdds,42,0.691406,0.691406,0.557026,0.845114,0.822378,0.880769,9.0,13.0,...,8.0,101.0,11.0,19.0,15.0,58.0,0.786896,0.663121,0.798754,0.029223
8,MultAdvBin4EqOdds,55,0.671875,0.671875,0.553043,0.784525,0.775971,0.801943,11.0,15.0,...,12.0,101.0,11.0,19.0,14.0,51.0,0.680569,0.659562,0.714729,-0.064427
9,MultAdvBin4EqOdds,73,0.730469,0.730469,0.613098,0.975327,0.988761,0.995704,9.0,21.0,...,9.0,108.0,14.0,25.0,9.0,56.0,0.843305,0.839575,0.904762,0.203552


In [12]:
result_df.to_csv(f'../../results/{data_name}-mult_adv-{epochs}.csv')